# # Film Eleştirileri ve Bag-of-Words Modellemesi

🎯 Bu zorluğun amacı, metinlerin ***Bag-of-words*** modellemesiyle oynamaktır.

✍️ Aşağıdaki veri setinde, _“olumlu”_ veya _“olumsuz”_ olarak sınıflandırılmış 2000 adet yorum bulunmaktadır.

In [1]:
import pandas as pd

data = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/movie_reviews.csv")
data.head()

,target,reviews
0,neg,"plot : two teen couples go to a church party ,..."
1,neg,the happy bastard's quick movie review \ndamn ...
2,neg,it is movies like these that make a jaded movi...
3,neg,""" quest for camelot "" is warner bros . ' firs..."
4,neg,synopsis : a mentally unstable man undergoing ...


In [2]:
data.shape

(2000, 2)

## 1. Ön işleme (Preprocessing)

❓ **Soru (Metin Temizleme)** ❓

- Bir cümleyi temizleyecek bir `preprocessing` fonksiyonu yazın ve bunu tüm yorumlara uygulayın. Fonksiyon şunları yapmalıdır:
    - boşlukları kaldırma
    - harfleri küçük harfe çevirme
    - sayıları kaldırma
    - noktalama işaretlerini kaldırma
    - tokenization (kelimelere ayırma)
    - lemmatization (kelime köküne indirgeme)
- Temizlenmiş yorumları `clean_reviews` adlı bir sütunda saklayabilirsiniz.
- Bu aşamada stopword’leri kaldırmayın; nedenini `3. N-gram modelleme` bölümünde açıklayacağız.

In [4]:
import string
import re
import nltk
from nltk.stem import WordNetLemmatizer

# Gerekli kaynakları indirelim
nltk.download('wordnet')
nltk.download('omw-1.4')

lemmatizer = WordNetLemmatizer()

def preprocessing(text):
    # 1. Küçük harfe çevirme ve gereksiz boşlukları (baş-son) kaldırma
    text = text.lower().strip()
    
    # 2. Sayıları kaldırma (Regex ile)
    text = re.sub(r'\d+', '', text)
    
    # 3. Noktalama işaretlerini kaldırma
    # string.punctuation içindeki her karakteri boşlukla değiştiriyoruz
    for char in string.punctuation:
        text = text.replace(char, '')
    
    # 4. Tokenization (Kelimelere ayırma)
    # Varsayılan split() metodu kelimeler arasındaki fazla boşlukları da temizler
    tokens = text.split()
    
    # 5. Lemmatization (Kelimeleri köklerine indirgeme)
    # Her bir token (kelime) için sözlük kökünü buluyoruz
    lemmatized_tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    # 6. Tekrar tek bir dize (string) haline getirme
    # (CountVectorizer gibi araçlar genellikle string girişi bekler)
    return " ".join(lemmatized_tokens)

# Fonksiyonu tüm yorumlara uygulayalım
data['clean_reviews'] = data['reviews'].apply(preprocessing)

# İlk 5 sonucu kontrol edelim
print("--- Ön İşleme Tamamlandı ---")
print(data[['reviews', 'clean_reviews']].head())

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/erdincuyar/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/erdincuyar/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


--- Ön İşleme Tamamlandı ---
                                             reviews  \
0  plot : two teen couples go to a church party ,...   
1  the happy bastard's quick movie review \ndamn ...   
2  it is movies like these that make a jaded movi...   
3   " quest for camelot " is warner bros . ' firs...   
4  synopsis : a mentally unstable man undergoing ...   

                                       clean_reviews  
0  plot two teen couple go to a church party drin...  
1  the happy bastard quick movie review damn that...  
2  it is movie like these that make a jaded movie...  
3  quest for camelot is warner bros first feature...  
4  synopsis a mentally unstable man undergoing ps...  


In [ ]:
# Yorumları temizle
pass  # SENİN KODUN BURAYA

❓ **Soru (LabelEncoding)**❓

Hedefinizi LabelEncode ile kodlayın ve `“target_encoded”` adlı bir sütuna kaydedin.

In [5]:
from sklearn.preprocessing import LabelEncoder

# 1. LabelEncoder nesnesini oluşturalım
le = LabelEncoder()

# 2. 'target' sütununu kodlayalım ve yeni sütuna kaydedelim
# Genellikle alfabetik sıraya göre (neg -> 0, pos -> 1) kodlama yapar
data['target_encoded'] = le.fit_transform(data['target'])

# 3. Hangi etiketin hangi sayıya denk geldiğini kontrol edelim
mapping = dict(zip(le.classes_, le.transform(le.classes_)))

print(f"Etiket Eşleşmeleri: {mapping}")
print("\n--- İlk 5 Satır (Kodlanmış) ---")
print(data[['target', 'target_encoded']].head())

Etiket Eşleşmeleri: {'neg': 0, 'pos': 1}

--- İlk 5 Satır (Kodlanmış) ---
  target  target_encoded
0    neg               0
1    neg               0
2    neg               0
3    neg               0
4    neg               0


In [6]:
# Hızlı kontrol
data.head()

,target,reviews,clean_reviews,target_encoded
0,neg,"plot : two teen couples go to a church party ,...",plot two teen couple go to a church party drin...,0
1,neg,the happy bastard's quick movie review \ndamn ...,the happy bastard quick movie review damn that...,0
2,neg,it is movies like these that make a jaded movi...,it is movie like these that make a jaded movie...,0
3,neg,""" quest for camelot "" is warner bros . ' firs...",quest for camelot is warner bros first feature...,0
4,neg,synopsis : a mentally unstable man undergoing ...,synopsis a mentally unstable man undergoing ps...,0


## 2. Bag-of-Words Modellemesi

❓ **Soru (Tek kelimelik sözcüklerle NaiveBayes)** ❓

`cross_validate` kullanarak, metinlerin Bag-of-Words temsilinde eğitilmiş bir Multinomial Naive Bayes modelini puanlayın.

In [7]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_validate

# 1. Metinleri Bag-of-Words (Unigram) formatına dönüştürelim
# Varsayılan olarak ngram_range=(1, 1) yani tek kelimeliktir.
vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(data['clean_reviews'])
y = data['target_encoded']

# 2. Multinomial Naive Bayes modelini tanımlayalım
nb_model = MultinomialNB()

# 3. cross_validate ile 5-katlı çapraz doğrulama yapalım
# 'accuracy' (doğruluk) skoruna odaklanıyoruz
cv_results = cross_validate(nb_model, X_bow, y, cv=5, scoring='accuracy')

# 4. Sonuçları yazdıralım
mean_score = cv_results['test_score'].mean()
print(f"Bag-of-Words (Unigram) Ortalama Doğruluk: %{mean_score * 100:.2f}")
print(f"Katlar Arası Standart Sapma: {cv_results['test_score'].std():.4f}")

Bag-of-Words (Unigram) Ortalama Doğruluk: %81.65
Katlar Arası Standart Sapma: 0.0111


## 3. N-gram Modellemesi

👀 Stop kelimeleri kaldırmamanızı istediğimizi hatırlayın. Neden? 

👉 Naive Bayes modelini bigramlarla eğiteceğiz. Bu nedenle, “I do not like coriander” (Kişnişi sevmiyorum) gibi bir cümlede, örneğin bu cümlede olumsuzluğu tespit etmek için “do not” bigramını taramak önemlidir.

❓ **Soru (bigramlarla NaiveBayes)** ❓

`cross_validate` kullanarak, metinlerin 2-gram Bag-of-Words temsilinde eğitilmiş bir Multinomial Naive Bayes modelini puanlayın.

In [8]:
vectorizer = CountVectorizer(ngram_range = (2,2))
naivebayes = MultinomialNB()

X_bow = vectorizer.fit_transform(data.clean_reviews)

cv_nb = cross_validate(
    naivebayes,
    X_bow,
    data.target_encoded,
    scoring = "accuracy"
)

round(cv_nb['test_score'].mean(),2)

0.84

🏁 Tebrikler! Artık vektörleştirilmiş metinler üzerinde Naive Bayes modelini nasıl eğiteceğinizi biliyorsunuz.

💾 Not defterinizi `git add/commit/push` yapmayı unutmayın...

